In [1]:
# Imports and repo setup for the adaptive LTL chain: first one of two targets, then drawer_closed, then the other target.
from pathlib import Path
from collections import Counter
import json
import os
import sys
import time

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")
os.environ.setdefault("XDG_CACHE_HOME", "/tmp")

import matplotlib.pyplot as plt
import numpy as np
import torch
from IPython.display import Image, display
from matplotlib.cm import ScalarMappable
from matplotlib.collections import LineCollection
from matplotlib.colors import Normalize

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / "calvin_experiments" / "calvin_rollout_utils.py").exists():
    if REPO_ROOT.parent == REPO_ROOT:
        raise RuntimeError("Could not find guided-diffusion repo root")
    REPO_ROOT = REPO_ROOT.parent

for path in [REPO_ROOT, REPO_ROOT / "robomimic", REPO_ROOT / "calvin" / "calvin_env", REPO_ROOT / "calvin_experiments"]:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

import robomimic.envs  # noqa: F401
import robomimic.utils.file_utils as FileUtils
import robomimic.utils.obs_utils as ObsUtils
import robomimic.utils.python_utils as PyUtils
import robomimic.utils.torch_utils as TorchUtils

from calvin_experiments import calvin_rollout_utils as CRU
from calvin_experiments.label_calvin_world_model import label_scene_states_for_names

print("repo:", REPO_ROOT)
print("cuda:", torch.cuda.is_available())


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


repo: /home/moritz/src/guided-diffusion
cuda: True


In [2]:
# Load the policy and automaton model. This LTL test uses the robust-direction initial scene and no local dynamics model.
DEVICE = TorchUtils.get_torch_device(try_to_use_cuda=True)

POLICY_CKPT_PATH = REPO_ROOT / "outputs/calvin/base_policy/calvin_D_base_dp/20260501015147/models/model_epoch_280.pth"
AUTOMATON_CKPT_PATH = REPO_ROOT / "outputs/calvin/automaton_world_model/h8_sh64_ah96_lh16_hh128_lr0.0003_epochs80_2026-05-05_20-38-38"
SCENE_CONFIG_PATH = REPO_ROOT / "calvin_experiments/configs/robust_direction_initial.json"
VISUALIZATION_CONFIG_PATH = REPO_ROOT / "calvin_experiments/configs/visualization_freiburg_style.json"
OUTPUT_ROOT = REPO_ROOT / "outputs/calvin/paper_stls/F_button_or_switch_then_drawer_then_remaining"
VIDEO_FPS = 30

policy, ckpt_dict = FileUtils.policy_from_checkpoint(ckpt_path=str(POLICY_CKPT_PATH), device=DEVICE, verbose=False)
policy_epoch = CRU.policy_epoch_from_checkpoint(POLICY_CKPT_PATH)
scene_cfg = CRU.load_json_config(SCENE_CONFIG_PATH)
video_cfg = CRU.load_json_config(VISUALIZATION_CONFIG_PATH)

automaton_model, automaton_stats, automaton_meta = CRU.automaton_model_for_eval(AUTOMATON_CKPT_PATH, DEVICE)
label_names = list(automaton_meta["label_names"])
label_thresholds = automaton_meta.get("label_thresholds")

OPPOSITE_LABEL_NAMES = {
    "switch_on": "switch_off", "switch_off": "switch_on",
    "button_on": "button_off", "button_off": "button_on",
    "drawer_open": "drawer_closed", "drawer_closed": "drawer_open",
    "door_left": "door_right", "door_right": "door_left",
}

def validate_target_label_idx(target_label_idx):
    target_label_idx = int(target_label_idx)
    if target_label_idx < 0 or target_label_idx >= len(label_names):
        raise ValueError(f"target_label_idx must be in [0, {len(label_names) - 1}], got {target_label_idx}")
    return target_label_idx

def label_idxs_from_names(target_names):
    missing = [name for name in target_names if name not in label_names]
    if missing:
        raise ValueError(f"Targets not in loaded automaton labels: {missing}; labels={label_names}")
    return [label_names.index(name) for name in target_names]

def opposite_label_idx(target_label_idx):
    target_label_idx = validate_target_label_idx(target_label_idx)
    opposite_name = OPPOSITE_LABEL_NAMES.get(label_names[target_label_idx])
    if opposite_name is None or opposite_name not in label_names:
        return None
    return label_names.index(opposite_name)

def score_label_probs(label_probs, target_label_idx):
    target_label_idx = validate_target_label_idx(target_label_idx)
    scores = np.asarray(label_probs, dtype=np.float32)[..., target_label_idx].copy()
    opp = opposite_label_idx(target_label_idx)
    if opp is not None:
        scores -= np.asarray(label_probs, dtype=np.float32)[..., opp]
    return scores

def score_rule_name(target_label_idx):
    target_label_idx = validate_target_label_idx(target_label_idx)
    opp = opposite_label_idx(target_label_idx)
    if opp is None:
        return f"p({label_names[target_label_idx]})"
    return f"p({label_names[target_label_idx]}) - p({label_names[opp]})"

print("device:", DEVICE)
print("scene config:", SCENE_CONFIG_PATH)
print("policy:", POLICY_CKPT_PATH)
print("automaton:", AUTOMATON_CKPT_PATH)
print("label order:")
for idx, name in enumerate(label_names):
    print(f"  {idx}: {name}  score={score_rule_name(idx)}")



============= Initialized Observation Utils with Obs Spec =============

using obs modality: low_dim with keys: ['proprio']
using obs modality: rgb with keys: ['third_person', 'eye_in_hand']
using obs modality: depth with keys: []
using obs modality: scan with keys: []
number of parameters: 6.931892e+07
device: cuda:0
scene config: /home/moritz/src/guided-diffusion/calvin_experiments/configs/robust_direction_initial.json
policy: /home/moritz/src/guided-diffusion/outputs/calvin/base_policy/calvin_D_base_dp/20260501015147/models/model_epoch_280.pth
automaton: /home/moritz/src/guided-diffusion/outputs/calvin/automaton_world_model/h8_sh64_ah96_lh16_hh128_lr0.0003_epochs80_2026-05-05_20-38-38
label order:
  0: switch_on  score=p(switch_on) - p(switch_off)
  1: switch_off  score=p(switch_off) - p(switch_on)
  2: button_on  score=p(button_on) - p(button_off)
  3: button_off  score=p(button_off) - p(button_on)
  4: button_pressed  score=p(button_pressed)
  5: drawer_open  score=p(drawer_open)

In [3]:
# Rollout helpers and automaton sample-and-rank utilities.
env = None
base_env_state = None

def current_automaton_state_and_label(env):
    state = env.get_state()
    robot = np.asarray(state["robot"], dtype=np.float32).reshape(-1)
    scene = np.asarray(state["scene"], dtype=np.float32).reshape(-1)
    automaton_state = np.concatenate([robot, scene]).astype(np.float32)
    automaton_label = label_scene_states_for_names(scene[None, :], label_names, label_thresholds)[0].astype(np.float32)
    return automaton_state, automaton_label

def repeat_obs_batch(obs_tensor, n):
    if int(n) == 1:
        return obs_tensor
    return {key: value.repeat((int(n),) + (1,) * (value.ndim - 1)) for key, value in obs_tensor.items()}

def unnormalize_action_sequence(action_sequence):
    action_np = CRU.to_numpy(action_sequence).astype(np.float32)
    if policy.action_normalization_stats is None:
        return action_np
    original_shape = action_np.shape
    flat_actions = action_np.reshape(-1, original_shape[-1])
    action_keys = policy.policy.global_config.train.action_keys
    action_shapes = {key: policy.action_normalization_stats[key]["offset"].shape[1:] for key in policy.action_normalization_stats}
    action_dict = PyUtils.vector_to_action_dict(flat_actions, action_shapes=action_shapes, action_keys=action_keys)
    action_dict = ObsUtils.unnormalize_dict(action_dict, normalization_stats=policy.action_normalization_stats)
    return PyUtils.action_dict_to_vector(action_dict, action_keys=action_keys).reshape(original_shape)

def predict_future_label_probs(automaton_state, automaton_label, action_chunks):
    action_chunks = np.asarray(action_chunks, dtype=np.float32)
    n_candidates, _, action_dim = action_chunks.shape
    automaton_horizon = len(automaton_stats["actions_mean"]) // action_dim
    scored_chunks = action_chunks[:, :automaton_horizon, :].reshape(n_candidates, -1)
    states = np.repeat(np.asarray(automaton_state, dtype=np.float32)[None, :], n_candidates, axis=0)
    labels = np.repeat(np.asarray(automaton_label, dtype=np.float32)[None, :], n_candidates, axis=0)
    states_t = torch.as_tensor((states - automaton_stats["states_mean"]) / automaton_stats["states_std"], device=DEVICE, dtype=torch.float32)
    actions_t = torch.as_tensor((scored_chunks - automaton_stats["actions_mean"]) / automaton_stats["actions_std"], device=DEVICE, dtype=torch.float32)
    labels_t = torch.as_tensor(labels, device=DEVICE, dtype=torch.float32)
    with torch.no_grad():
        probs = torch.sigmoid(automaton_model(states_t, actions_t, labels_t)).detach().cpu().numpy()
    return probs, automaton_horizon

def rollout_policy_once(seed, action_sampler, output_dir, rollout_tag, horizon=500, save_video=True, stop_when_complete=True):
    global env, base_env_state
    CRU.drop_env_quietly(globals(), name="env")
    env, base_env_state = CRU.load_fresh_env_from_checkpoint(ckpt_dict, seed=int(seed), suppress_output=True)
    fixed_scene, fixed_robot, active_scene_cfg = CRU.fixed_scene_robot_from_config(base_env_state, SCENE_CONFIG_PATH)

    CRU.seed_everything(seed)
    policy.start_episode()
    obs = CRU.reset_env_to_scene_robot(env, fixed_scene, fixed_robot)
    scene_snapshot = CRU.capture_scene_snapshot(env)
    frames = [CRU.render_visual_camera(env, video_cfg)] if save_video else []

    start_state = env.get_state()
    start_scene = np.asarray(start_state["scene"], dtype=np.float32).copy()
    binaries = CRU.articulated_binaries_from_start_state(start_scene)
    _, label0 = current_automaton_state_and_label(env)
    if hasattr(action_sampler, "sync"):
        action_sampler.sync(label0, 0)

    actions, rewards, dones, records = [], [], [], []
    target_events = []
    scene_states = [start_scene.copy()]
    robot_states = [np.asarray(start_state["robot"], dtype=np.float32).copy()]
    eef_xy = [robot_states[-1][:2].copy()]
    action_queue = []
    first_behavior, first_behavior_step = "none", -1
    termination_reason = "horizon"

    for step in range(int(horizon)):
        if not action_queue:
            new_actions, record = action_sampler(obs, env, step)
            action_queue.extend(np.asarray(new_actions, dtype=np.float32))
            records.append(record)
        action = np.asarray(action_queue.pop(0), dtype=np.float32).copy()
        actions.append(action.copy())

        obs, reward, done, _ = env.step(action)
        state = env.get_state()
        scene = np.asarray(state["scene"], dtype=np.float32).copy()
        robot = np.asarray(state["robot"], dtype=np.float32).copy()
        _, current_label = current_automaton_state_and_label(env)

        rewards.append(float(reward)); dones.append(bool(done))
        scene_states.append(scene); robot_states.append(robot); eef_xy.append(robot[:2].copy())
        if save_video:
            frames.append(CRU.render_visual_camera(env, video_cfg))

        if first_behavior_step < 0 and CRU.check_state_difference(start_scene, scene, robot[:3], binaries, for_display=False):
            first_behavior = CRU.classify_behavior(start_scene, scene, robot[:3], binaries, for_display=False)
            first_behavior_step = step + 1

        if hasattr(action_sampler, "sync"):
            events_before = len(action_sampler.events)
            advanced = action_sampler.sync(current_label, step + 1)
            if advanced:
                action_queue.clear()
                target_events.extend(action_sampler.events[events_before:])
            if stop_when_complete and action_sampler.done:
                termination_reason = "ltl_complete"
                break
        if done:
            termination_reason = "env_done"
            break

    _, labelf = current_automaton_state_and_label(env)
    ltl_complete = bool(getattr(action_sampler, "done", False))
    behavior = "ltl_complete" if ltl_complete else first_behavior
    rollout = {
        "scene_config": active_scene_cfg["name"], "seed": int(seed), "behavior": behavior,
        "first_behavior": first_behavior, "first_behavior_step": int(first_behavior_step),
        "behavior_step": int(target_events[-1]["step"] if target_events else first_behavior_step),
        "termination_step": len(actions), "termination_reason": termination_reason,
        "return": float(np.sum(rewards)), "actions": np.asarray(actions, dtype=np.float32),
        "rewards": np.asarray(rewards, dtype=np.float32), "dones": np.asarray(dones, dtype=bool),
        "scene_states": np.asarray(scene_states, dtype=np.float32), "robot_states": np.asarray(robot_states, dtype=np.float32),
        "eef_xy": np.asarray(eef_xy, dtype=np.float32), "initial_label": label0.astype(int).tolist(),
        "final_label": labelf.astype(int).tolist(), "records": records, "target_events": target_events,
        "first_choice_name": getattr(action_sampler, "first_choice_name", None),
        "remaining_first_name": getattr(action_sampler, "remaining_first_name", None),
        "ltl_complete": ltl_complete, "scene_snapshot": scene_snapshot,
    }
    if save_video:
        CRU.save_rollout_artifacts(rollout, frames, output_dir, rollout_tag, video_cfg, fps=VIDEO_FPS)
    return rollout


In [4]:
# Adaptive LTL sampler: first satisfy one OR option, then drawer_closed, then satisfy the remaining OR option.
def make_or_then_middle_then_remaining_action_provider(first_option_names, middle_target_name, n_candidates):
    first_option_idxs = label_idxs_from_names(first_option_names)
    middle_idx = label_idxs_from_names([middle_target_name])[0]
    n_candidates = int(n_candidates)
    state = {
        "phase": "first_or",
        "events": [],
        "first_choice_idx": None,
        "remaining_first_idxs": [],
        "last_selected_option_idx": None,
    }

    def mark_event(step, role, target_idx):
        event = {
            "step": int(step),
            "role": role,
            "target_idx": int(target_idx),
            "target_name": label_names[int(target_idx)],
        }
        state["events"].append(event)
        return event

    def expose_state():
        action_provider.phase = state["phase"]
        action_provider.events = state["events"]
        action_provider.done = state["phase"] == "done"
        action_provider.first_choice_name = None if state["first_choice_idx"] is None else label_names[state["first_choice_idx"]]
        action_provider.remaining_first_name = None if not state["remaining_first_idxs"] else label_names[state["remaining_first_idxs"][0]]

    def sync(label, step):
        advanced = False
        while state["phase"] != "done":
            if state["phase"] == "first_or":
                achieved = [idx for idx in first_option_idxs if float(label[idx]) > 0.5]
                if not achieved:
                    break
                last_selected = state.get("last_selected_option_idx")
                chosen = last_selected if last_selected in achieved else achieved[0]
                state["first_choice_idx"] = int(chosen)
                state["remaining_first_idxs"] = [idx for idx in first_option_idxs if idx != chosen]
                mark_event(step, "first_or", chosen)
                state["phase"] = "middle"
                advanced = True
                continue

            if state["phase"] == "middle":
                if float(label[middle_idx]) <= 0.5:
                    break
                mark_event(step, "middle", middle_idx)
                state["phase"] = "remaining_first" if state["remaining_first_idxs"] else "done"
                advanced = True
                continue

            if state["phase"] == "remaining_first":
                current_remaining = state["remaining_first_idxs"][0]
                if float(label[current_remaining]) <= 0.5:
                    break
                mark_event(step, "remaining_first", current_remaining)
                state["remaining_first_idxs"].pop(0)
                state["phase"] = "remaining_first" if state["remaining_first_idxs"] else "done"
                advanced = True
                continue

        expose_state()
        return advanced

    def current_target_idxs():
        if state["phase"] == "first_or":
            return list(first_option_idxs)
        if state["phase"] == "middle":
            return [middle_idx]
        if state["phase"] == "remaining_first":
            return list(state["remaining_first_idxs"])
        return [middle_idx]

    def action_provider(obs, env, step):
        automaton_state, automaton_label = current_automaton_state_and_label(env)
        sync(automaton_label, step)
        target_idxs = current_target_idxs()

        obs_tensor = policy._prepare_observation(obs)
        obs_tensor_rank = repeat_obs_batch(obs_tensor, n_candidates)
        with torch.no_grad():
            action_chunk_n = policy.policy._get_action_trajectory(obs_dict=obs_tensor_rank)
        action_chunks = unnormalize_action_sequence(action_chunk_n)
        candidate_probs, automaton_horizon = predict_future_label_probs(automaton_state, automaton_label, action_chunks)

        target_score_columns = np.stack([score_label_probs(candidate_probs, idx) for idx in target_idxs], axis=1)
        candidate_scores = target_score_columns.max(axis=1)
        candidate_target_pos = target_score_columns.argmax(axis=1)
        selected_idx = int(np.argmax(candidate_scores))
        selected_target_pos = int(candidate_target_pos[selected_idx])
        selected_target_idx = int(target_idxs[selected_target_pos])
        state["last_selected_option_idx"] = selected_target_idx if state["phase"] == "first_or" else state.get("last_selected_option_idx")
        selected_opp = opposite_label_idx(selected_target_idx)
        expose_state()

        record = {
            "t": int(step),
            "phase": state["phase"],
            "first_option_names": list(first_option_names),
            "middle_target_name": middle_target_name,
            "first_choice_name": action_provider.first_choice_name,
            "remaining_first_name": action_provider.remaining_first_name,
            "target_names": [label_names[idx] for idx in target_idxs],
            "target_score_rules": [score_rule_name(idx) for idx in target_idxs],
            "current_label": automaton_label.astype(int).tolist(),
            "selected_idx": selected_idx,
            "selected_score": float(candidate_scores[selected_idx]),
            "selected_target_idx": selected_target_idx,
            "selected_target_name": label_names[selected_target_idx],
            "selected_target_score": float(target_score_columns[selected_idx, selected_target_pos]),
            "selected_target_prob": float(candidate_probs[selected_idx, selected_target_idx]),
            "selected_opposite_idx": None if selected_opp is None else int(selected_opp),
            "selected_opposite_name": None if selected_opp is None else label_names[selected_opp],
            "selected_opposite_prob": None if selected_opp is None else float(candidate_probs[selected_idx, selected_opp]),
            "selected_target_scores": {label_names[idx]: float(target_score_columns[selected_idx, pos]) for pos, idx in enumerate(target_idxs)},
            "pred_probs": candidate_probs[selected_idx].tolist(),
            "candidate_scores": candidate_scores.tolist(),
        }
        return np.asarray(action_chunks[selected_idx, :automaton_horizon, :], dtype=np.float32), record

    action_provider.sync = sync
    expose_state()
    return action_provider


In [5]:
# Experiment knobs and rollout loop. The final target is chosen adaptively as the first OR option that was not achieved.
FIRST_OPTION_NAMES = ["button_pressed", "switch_off"]
MIDDLE_TARGET_NAME = "drawer_closed"
SEEDS = [0, 1, 2, 3, 4]
N_CANDIDATES = 32
HORIZON = 500
SAVE_VIDEO = True

FORMULA_TEXT = "F((button_pressed OR switch_off) ; drawer_closed ; remaining first option)"
BRANCH_FORMULA_TEXT = "F(button_pressed & F(drawer_closed & F(switch_off))) OR F(switch_off & F(drawer_closed & F(button_pressed)))"

run_dir = OUTPUT_ROOT / time.strftime("%Y%m%d_%H%M%S")
run_dir.mkdir(parents=True, exist_ok=False)

settings = {
    "formula": FORMULA_TEXT,
    "branch_formula": BRANCH_FORMULA_TEXT,
    "first_option_names": FIRST_OPTION_NAMES,
    "middle_target_name": MIDDLE_TARGET_NAME,
    "scene_config_path": str(SCENE_CONFIG_PATH),
    "scene_config_name": scene_cfg.get("name"),
    "seeds": [int(s) for s in SEEDS],
    "n_candidates": int(N_CANDIDATES),
    "horizon": int(HORIZON),
    "score_rules": {name: score_rule_name(label_names.index(name)) for name in FIRST_OPTION_NAMES + [MIDDLE_TARGET_NAME]},
}
print(json.dumps(settings, indent=2))

rollouts = []
for seed in SEEDS:
    action_sampler = make_or_then_middle_then_remaining_action_provider(FIRST_OPTION_NAMES, MIDDLE_TARGET_NAME, N_CANDIDATES)
    rollout = rollout_policy_once(
        seed=seed,
        action_sampler=action_sampler,
        output_dir=run_dir,
        rollout_tag=f"adaptive_or_then_{MIDDLE_TARGET_NAME}_then_remaining_seed_{seed:03d}",
        horizon=HORIZON,
        save_video=SAVE_VIDEO,
        stop_when_complete=True,
    )
    rollouts.append(rollout)
    event_str = ", ".join(f"{e['role']}:{e['target_name']}@{e['step']}" for e in rollout["target_events"])
    print(
        f"seed {seed:03d}: complete={rollout['ltl_complete']}, steps={rollout['termination_step']:>3}, "
        f"first_choice={rollout['first_choice_name']}, remaining={rollout['remaining_first_name']}, "
        f"events=[{event_str}], video={rollout.get('video')}"
    )

summary = {
    **settings,
    "n_rollouts": len(rollouts),
    "ltl_complete_rate": sum(r["ltl_complete"] for r in rollouts) / max(1, len(rollouts)),
    "first_choice_counts": dict(Counter(r["first_choice_name"] for r in rollouts)),
    "event_sequence_counts": dict(Counter(" -> ".join(e["target_name"] for e in r["target_events"]) for r in rollouts)),
    "behavior_counts": dict(Counter(r["behavior"] for r in rollouts)),
    "first_behavior_counts": dict(Counter(r["first_behavior"] for r in rollouts)),
    "rollouts": [
        {
            "seed": r["seed"], "ltl_complete": r["ltl_complete"], "termination_step": r["termination_step"],
            "termination_reason": r["termination_reason"], "first_choice_name": r["first_choice_name"],
            "remaining_first_name": r["remaining_first_name"], "first_behavior": r["first_behavior"],
            "first_behavior_step": r["first_behavior_step"], "target_events": r["target_events"],
            "initial_label": r["initial_label"], "final_label": r["final_label"],
            "video": None if "video" not in r else str(r["video"]), "trace": None if "trace" not in r else str(r["trace"]),
        }
        for r in rollouts
    ],
}
(run_dir / "summary.json").write_text(json.dumps(summary, indent=2))
print(json.dumps(summary, indent=2))
print("run_dir:", run_dir)


{
  "formula": "F((button_pressed OR switch_off) ; drawer_closed ; remaining first option)",
  "branch_formula": "F(button_pressed & F(drawer_closed & F(switch_off))) OR F(switch_off & F(drawer_closed & F(button_pressed)))",
  "first_option_names": [
    "button_pressed",
    "switch_off"
  ],
  "middle_target_name": "drawer_closed",
  "scene_config_path": "/home/moritz/src/guided-diffusion/calvin_experiments/configs/robust_direction_initial.json",
  "scene_config_name": "robust_direction_initial",
  "seeds": [
    0,
    1,
    2,
    3,
    4
  ],
  "n_candidates": 32,
  "horizon": 500,
  "score_rules": {
    "button_pressed": "p(button_pressed)",
    "switch_off": "p(switch_off) - p(switch_on)",
    "drawer_closed": "p(drawer_closed) - p(drawer_open)"
  }
}
ven = NVIDIA Corporation
ven = NVIDIA Corporation


/home/moritz/src/guided-diffusion/.pixi/envs/default/lib/python3.10/site-packages/gym/utils/passive_env_checker.py:174: UserWarning: WARN: Future gym versions will require that `Env.reset` can be passed a `seed` instead of using `Env.seed` for resetting the environment random number generator.
  logger.warn(
/home/moritz/src/guided-diffusion/.pixi/envs/default/lib/python3.10/site-packages/gym/utils/passive_env_checker.py:187: UserWarning: WARN: Future gym versions will require that `Env.reset` can be passed `options` to allow the environment initialisation to be passed additional information.
  logger.warn(
/home/moritz/src/guided-diffusion/.pixi/envs/default/lib/python3.10/site-packages/gym/utils/passive_env_checker.py:195: UserWarning: WARN: The result returned by `env.reset()` was not a tuple of the form `(obs, info)`, where `obs` is a observation and `info` is a dictionary containing additional information. Actual type: `<class 'dict'>`
  logger.warn(


ObservationKeyToModalityDict: states not found, adding states to mapping with assumed low_dim modality!
ObservationKeyToModalityDict: timesteps not found, adding timesteps to mapping with assumed low_dim modality!
ObservationKeyToModalityDict: actions not found, adding actions to mapping with assumed low_dim modality!


/home/moritz/src/guided-diffusion/.pixi/envs/default/lib/python3.10/site-packages/gym/utils/passive_env_checker.py:219: DeprecationWarning: WARN: Core environment is written in old step API which returns one bool instead of two. It is recommended to rewrite the environment with new step API. 
  logger.deprecation(
/home/moritz/src/guided-diffusion/.pixi/envs/default/lib/python3.10/site-packages/gym/utils/passive_env_checker.py:225: DeprecationWarning: `np.bool8` is a deprecated alias for `np.bool_`.  (Deprecated NumPy 1.24)
  if not isinstance(done, (bool, np.bool8)):
/home/moritz/src/guided-diffusion/.pixi/envs/default/lib/python3.10/site-packages/gym/utils/passive_env_checker.py:137: UserWarning: WARN: The obs returned by the `step()` method was expecting a numpy array, actual type: <class 'dict'>
  logger.warn(
/home/moritz/src/guided-diffusion/.pixi/envs/default/lib/python3.10/site-packages/gym/spaces/box.py:227: UserWarning: WARN: Casting input x to numpy array.
  logger.warn("Cas

seed 000: complete=True, steps=196, first_choice=button_pressed, remaining=None, events=[first_or:button_pressed@34, middle:drawer_closed@122, remaining_first:switch_off@196], video=/home/moritz/src/guided-diffusion/outputs/calvin/paper_stls/F_button_or_switch_then_drawer_then_remaining/20260521_185831/adaptive_or_then_drawer_closed_then_remaining_seed_000/robust_direction_initial_adaptive_or_then_drawer_closed_then_remaining_seed_000_freiburg_style.mp4
ven = NVIDIA Corporation
ven = NVIDIA Corporation
seed 001: complete=False, steps=500, first_choice=None, remaining=None, events=[], video=/home/moritz/src/guided-diffusion/outputs/calvin/paper_stls/F_button_or_switch_then_drawer_then_remaining/20260521_185831/adaptive_or_then_drawer_closed_then_remaining_seed_001/robust_direction_initial_adaptive_or_then_drawer_closed_then_remaining_seed_001_freiburg_style.mp4
ven = NVIDIA Corporation
ven = NVIDIA Corporation
seed 002: complete=True, steps=193, first_choice=button_pressed, remaining=No

In [ ]:
# Plot all rollout paths with temporal color and event markers; inspect which target was selected per chunk.
def plot_rollout_xy_time_gradient(rollouts, scene_snapshot, title, save_path=None, cmap_name="plasma"):
    paths = [np.asarray(r["eef_xy"], dtype=np.float32) for r in rollouts]
    max_step = max(max(1, len(path) - 1) for path in paths)
    norm = Normalize(vmin=0, vmax=max_step)
    cmap = plt.get_cmap(cmap_name)

    fig, ax = plt.subplots(figsize=(9, 7))
    ax.set_facecolor("#fbf7ef")
    CRU.draw_scene_snapshot(ax, scene_snapshot)

    for rollout, xy in zip(rollouts, paths):
        if len(xy) >= 2:
            segments = np.stack([xy[:-1], xy[1:]], axis=1)
            lc = LineCollection(segments, cmap=cmap, norm=norm, linewidth=2.3, alpha=0.82)
            lc.set_array(np.arange(len(segments), dtype=np.float32))
            ax.add_collection(lc)
        ax.scatter(xy[0, 0], xy[0, 1], c="white", s=46, edgecolors="black", linewidths=0.9, zorder=5)
        ax.scatter(xy[-1, 0], xy[-1, 1], c="black", s=42, edgecolors="white", linewidths=0.7, zorder=5)
        ax.text(xy[-1, 0] + 0.004, xy[-1, 1] + 0.004, f"s{rollout['seed']}", fontsize=8, color="black")

        for event in rollout.get("target_events", []):
            step = min(int(event["step"]), len(xy) - 1)
            ax.scatter(xy[step, 0], xy[step, 1], marker="*", s=115, c=[cmap(norm(step))], edgecolors="black", linewidths=0.7, zorder=6)
            ax.text(xy[step, 0] + 0.004, xy[step, 1] - 0.010, f"{event['role']}\n{event['target_name']}", fontsize=7, color="black")

    all_xy = np.concatenate(paths, axis=0)
    xlim, ylim = CRU.scene_limits_from_snapshot(scene_snapshot, all_xy)
    ax.set_xlim(*xlim); ax.set_ylim(*ylim)
    ax.set_title(title)
    ax.set_xlabel("world x [m]"); ax.set_ylabel("world y [m]")
    ax.grid(alpha=0.3)
    cbar = fig.colorbar(ScalarMappable(norm=norm, cmap=cmap), ax=ax, fraction=0.035, pad=0.02)
    cbar.set_label("rollout step")
    if save_path is not None:
        fig.savefig(save_path, dpi=180, bbox_inches="tight")
        print("plot:", save_path)
        plt.close(fig)
        display(Image(filename=str(save_path)))
    else:
        plt.show()

plot_rollout_xy_time_gradient(
    rollouts,
    rollouts[0]["scene_snapshot"],
    title=f"Adaptive LTL sample-and-rank | N={len(rollouts)}, candidates={N_CANDIDATES}\n{FORMULA_TEXT}",
    save_path=run_dir / "rollouts_xy_time_gradient.png",
)

rows = []
for rollout in rollouts:
    for record in rollout.get("records", []):
        rows.append({
            "seed": rollout["seed"], "t": record["t"], "phase": record["phase"],
            "target_names": record["target_names"], "selected_target": record["selected_target_name"],
            "score": record["selected_score"], "target_scores": record["selected_target_scores"],
            "target_prob": record["selected_target_prob"], "opposite_prob": record["selected_opposite_prob"],
        })

print("first selected chunks:")
for row in rows[:25]:
    print(row)

CRU.drop_env_quietly(globals(), name="env")
